# Thesis figures — SP vs TS

Every figure is produced from the saved final outputs and written to `V1_Drift/thesis_figures/` as **PNG (300 dpi) + PDF**, with a **CSV** of every plotted number and a **JSON** file recording the configuration and statistics (model, normalization, voxel rule, aggregation, drift statistic, test) — so each figure is traceable as required in THESIS_CURRENT_STATE.

| Figure | Content | Inputs |
|---|---|---|
| 1 | Whole-data validation R², SP vs TS (k1), Henderson et al. as reference | whole-data fits |
| S1 | Implementation check against Henderson et al. (2023) | whole-data fits |
| 2 | Single-voxel drift: cvR² and r vs session lag; drift strength per ROI (all voxels) | session-wise fits |
| 3 | Population response drift, SP vs TS (all voxels; SP with matched voxel count as supplement) | SP notebook 6, TS `save_perm_pop_TS` |
| 4 | RDM drift: 1−r/Spearman (geometry) and Euclidean/RMSE (distance scale) | same |
| 5 | SP controls (single voxel): matched voxel count, 4 × 6 SF levels, gray-background images; SP population/RDM with matched voxel count as supplement | SP session-wise fits |

Conventions: SP blue, TS orange in every figure; curves mean ± SEM across subjects; bars mean, dots subjects; drift strength = lag correlation signed so that positive = more drift; stars = session-order permutation test (1000), FDR-BH across the 4 ROIs per model. Missing inputs are skipped and reported (e.g. TS population before `save_perm_pop_TS` has run). Loading the single-voxel results takes several minutes the first time; summaries are cached in `thesis_figures/cache/` (set `RECOMPUTE = True` after new runs).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==== Configuration ====
import os, glob, json, pickle, time, warnings
import numpy as np, h5py, matplotlib as mpl, matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- inputs (read-only) ----
FITS     = "/content/drive/MyDrive/V2_drift/texturemodel/model_fits/"             # TS fits (whole-data + FAST_CROSS_SESSION)
ROIS_DIR = "/content/drive/MyDrive/V2_drift/nsd/rois/"                            # Henderson ROI info + noise ceilings
NSD      = "/content/drive/MyDrive/V1_Drift/NSD/"                                 # prf-visualrois, pRF R2
SP_SESS  = "/content/drive/MyDrive/V1_Drift/repDrift_expand/"                     # SP session-wise + permPop (+ matchVox/)
SP_WHOLE = SP_SESS + "wholedata/"
SP_SESS_6LEV = "/content/drive/MyDrive/V1_Drift/repDrift_expand_6levels/"              # SP control: 4 ori x 6 SF
SP_SESS_BG   = "/content/drive/MyDrive/V1_Drift/repDrift_expand_with_background/"     # SP control: image on gray background                                                 # SP whole-data fits
TS_POP   = "/content/drive/MyDrive/V2_drift/texturemodel/drift_results/population_permutations/"  # TS permPop
TS_EXCL  = "exclAbs100/"   # TS voxels with |simulated response| > 100% excluded (save_perm_pop_TS EXCL_ABS_PCT); "" = none

# ---- outputs ----
OUT   = "/content/drive/MyDrive/V1_Drift/thesis_figures/"    # PNG (300 dpi) + PDF + CSV (plotted numbers) + JSON (configuration)
CACHE = OUT + "cache/"                                        # computed summaries; delete a file to recompute it

# ---- analysis settings (final configuration) ----
ROIS     = ["V1", "V2", "V3", "hV4"]
CODES    = {"V1": [1, 2], "V2": [3, 4], "V3": [5, 6], "hV4": [7]}
SP_VREG  = {"V1": 1, "V2": 2, "V3": 3, "hV4": 4}
SUBJECTS = list(range(1, 9))
NSESS, NPERM, MATCH_SEED = 30, 1000, 0
TS_TAG   = "k1"                                               # TS configuration: 49 LL + 1 PC per HL group (58 predictors)
SP_POP_ZSTR = ""                                              # normalization used for SP notebook 6 runs ("" = raw)

# ---- style: one colour per model, used in every figure (validated palette, CVD-safe) ----
COL = {"SP": "#257de0", "TS": "#f47230",                       # models
       "Henderson": "#6a4ce2", "TS_herpRF": "#00b894",         # validation figure only
       "SP 6 levels": "#00b894", "SP background": "#6a4ce2", "SP matched voxels": "#d1416f"}    # SP-control figure only (SP 4x4 stays blue)
LABEL = {"SP": "SP (4 ori × 4 SF, 18)", "TS": "TS (k1, 58)"}
mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7.5, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.6,
    "axes.edgecolor": "#444444", "xtick.color": "#444444", "ytick.color": "#444444",
    "axes.labelcolor": "#222222", "text.color": "#222222", "savefig.dpi": 300, "pdf.fonttype": 42,
})
W_FULL = 6.3                                                    # inches, A4 text width (16 cm)
os.makedirs(CACHE, exist_ok=True)
print("figures ->", OUT)


In [ ]:
# ==== Shared helpers: statistics, saving, small plotting utilities ====
lag = np.abs(np.subtract.outer(np.arange(NSESS), np.arange(NSESS))); OFF = lag > 0
_rng = np.random.default_rng(0); PERMS = [_rng.permutation(NSESS) for _ in range(NPERM)]

def lag_corr(G):
    """Roth & Merriam: Pearson r between a session x session matrix and session lag (off-diagonal entries)."""
    return np.corrcoef(G[OFF], lag[OFF])[0, 1]

def lag_curve(G):
    return np.array([np.nanmean(G[lag == d]) for d in range(1, NSESS)])

def group_test(Gs, sign):
    """Drift strength per subject (sign * r; positive = more drift), group mean, one-sided permutation p
    (session order permuted identically for all subjects)."""
    subj = np.array([sign * lag_corr(G) for G in Gs])
    null = np.array([np.mean([sign * lag_corr(G[np.ix_(p, p)]) for G in Gs]) for p in PERMS])
    obs = subj.mean()
    return dict(subj=subj, mean=obs, p=(np.sum(null >= obs) + 1) / (len(null) + 1))

def fdr_bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); q[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    return np.minimum(q, 1)

def stars(q):
    return "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "n.s."

def sem(a, axis=0):
    a = np.asarray(a, float); return np.nanstd(a, axis=axis, ddof=1) / np.sqrt(np.sum(np.isfinite(a), axis=axis))

def save_fig(fig, name, meta, rows):
    """PNG (300 dpi) + PDF + CSV of every plotted number + JSON with the full configuration (traceability)."""
    os.makedirs(OUT, exist_ok=True)
    fig.savefig(OUT + name + ".png", bbox_inches="tight"); fig.savefig(OUT + name + ".pdf", bbox_inches="tight")
    if rows:
        keys = list(rows[0].keys())
        with open(OUT + name + ".csv", "w") as f:
            f.write(",".join(keys) + "\n")
            for r in rows: f.write(",".join(str(r[k]) for k in keys) + "\n")
    meta = dict(meta, created=time.strftime("%Y-%m-%d %H:%M"), subjects=SUBJECTS, rois=ROIS)
    with open(OUT + name + ".json", "w") as f: json.dump(meta, f, indent=1, default=str)
    print("saved:", OUT + name + ".{png,pdf,csv,json}")

def bars_with_subjects(ax, xs, values_by_series, colors, labels, width=None, hatch=None):
    """Bar = mean across subjects, error bar = SEM, dots = subjects (no colour-only identity: legend + labels)."""
    k = len(values_by_series); width = width or 0.8 / k
    for j, (vals, c, lab) in enumerate(zip(values_by_series, colors, labels)):
        pos = np.asarray(xs) + (j - (k - 1) / 2) * width
        mu = [np.nanmean(v) if v is not None else np.nan for v in vals]
        er = [sem(v) if v is not None else np.nan for v in vals]
        ax.bar(pos, mu, width * 0.88, color=c, label=lab, edgecolor="white", linewidth=0.8, zorder=2)
        ax.errorbar(pos, mu, yerr=er, fmt="none", ecolor="#222222", elinewidth=0.9, capsize=0, zorder=3)
        for p_, v in zip(pos, vals):
            if v is None: continue
            jit = np.linspace(-width * 0.22, width * 0.22, len(v))
            ax.plot(p_ + jit, v, "o", ms=2.6, mfc="white", mec="#444444", mew=0.6, zorder=4)
    ax.axhline(0, color="#444444", lw=0.6, zorder=1)
    ax.set_xticks(xs); ax.set_xticklabels(ROIS)
    return [np.asarray(xs) + (j - (k - 1) / 2) * width for j in range(k)]

RECOMPUTE = False      # True: ignore cached summaries (e.g. after new runs finished)

def cached(name, fn):
    path = CACHE + name + ".pkl"
    if os.path.exists(path) and not RECOMPUTE:
        with open(path, "rb") as f: print("loaded cache:", path); return pickle.load(f)
    out = fn()
    with open(path, "wb") as f: pickle.dump(out, f)
    return out


In [ ]:
# ==== Data loaders (each result is cached in OUT/cache/) ====
import nibabel as nib

def flat(v): v = v[0] if isinstance(v, (list, tuple)) else v; return np.asarray(v).ravel().astype(int)
def col0(x): x = np.asarray(x); return x[:, 0] if x.ndim > 1 else x

# ---------- whole-data validation R2 (Henderson Fig. 3B voxels: NC > 0.01, no category-ROI overlap) ----------
def _my_fit(s, roi, folder, n_ok):
    for fn in sorted(glob.glob(FITS + f"S{s:02d}/" + folder.format(roi=roi) + "/*/all_fit_params.npy"),
                     key=os.path.getmtime, reverse=True):
        d = np.load(fn, allow_pickle=True).item()
        if n_ok(np.asarray(d["best_params"][1]).shape[1]): return dict(zip(flat(d["voxel_index"]), col0(d["val_r2"])))
def _her_fit(s, roi):
    fn = FITS + f"MAGGIES/S{s:02d}/all_fit_params.npy"
    if os.path.exists(fn):
        d = np.load(fn, allow_pickle=True).item(); return dict(zip(flat(d["voxel_index"]), col0(d["val_r2"])))
def _sp_fit(s, roi):
    fn = SP_WHOLE + f"SP_wholedata_sub{s}.npy"
    if os.path.exists(fn):
        d = np.load(fn, allow_pickle=True).item()[roi]; return dict(zip(d["voxel_index"], d["val_r2"]))
WHOLE_MODELS = {
    "Henderson et al. (2023), TS":      _her_fit,
    "TS, Henderson pRFs (validation)":  lambda s, r: _my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_pca95_use_henderson_grid5_pRFs", lambda n: n > 60),
    "TS k1 (58), NSD pRFs":             lambda s, r: _my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k1_use_nsd_grid5_pRFs", lambda n: n == 58),
    "TS LL only (49), NSD pRFs":        lambda s, r: _my_fit(s, r, "texture_pyramid_OLS_4ori_4sf_pcaHL_HendersonGroups_{roi}_k1_LLonly_use_nsd_grid5_pRFs", lambda n: n == 49),
    "SP (18), NSD pRFs":                _sp_fit,
}
def load_wholedata():
    out = {m: np.full((len(SUBJECTS), 4), np.nan) for m in WHOLE_MODELS}
    for si, s in enumerate(SUBJECTS):
        info = np.load(ROIS_DIR + f"S{s}_voxel_roi_info.npy", allow_pickle=True).item()
        vidx = flat(info["voxel_idx"]); nc = np.load(ROIS_DIR + f"S{s}_noise_ceiling_avgreps.npy") / 100
        ret, face, place, body = (np.asarray(info[k])[vidx] for k in ["roi_labels_retino", "roi_labels_face", "roi_labels_place", "roi_labels_body"])
        categ = np.isin(face, [2, 3]) | np.isin(place, [1, 2, 3]) | np.isin(body, [1])
        for ri, roi in enumerate(ROIS):
            sel = vidx[np.isin(ret, CODES[roi]) & ~categ & (nc > 0.01)]
            for m, load in WHOLE_MODELS.items():
                lk = load(s, roi)
                if lk is None: continue
                v = np.array([lk.get(x, np.nan) for x in sel], float)
                if not m.startswith("Henderson"): v[v < -1] = np.nan      # OLS-unstable voxels excluded (counted in the log)
                out[m][si, ri] = np.nanmean(v)
    return out

# ---------- single-voxel cross-session matrices (Roth & Merriam: pRF R2 > 0 voxels, median per session pair) ----------
def voxel_sets(s, match):
    ld = lambda f: nib.load(NSD + f"subj{s:02d}/func1pt8mm/{f}").get_fdata().ravel()
    vis, pr2 = ld("roi/prf-visualrois.nii.gz"), ld("R2.nii.gz")
    goods = {roi: np.isin(vis, CODES[roi]) & (pr2 > 0) for roi in ROIS}
    if match:
        n_min = min(int(g.sum()) for g in goods.values())
        for ri, roi in enumerate(ROIS):
            idx = np.random.default_rng([MATCH_SEED, s, ri]).choice(np.flatnonzero(goods[roi]), n_min, replace=False)
            goods[roi] = np.zeros_like(goods[roi]); goods[roi][idx] = True
    return goods

def _ts_sv(s, roi, good):
    fns = glob.glob(FITS + f"S{s:02d}/FAST_CROSS_SESSION/S{s:02d}_{roi}_texture_pyramid_OLS_4ori_4sf_"
                    f"pcaHL_HendersonGroups_{roi}_{TS_TAG}_use_nsd_grid5_pRFs_*cross_session.h5py")
    if not fns: return None
    with h5py.File(fns[0], "r") as f:
        keep = good[np.asarray(f["voxel_index"]).ravel()]
        return {k: np.nanmedian(np.asarray(f[k][:NSESS, :NSESS, :, 0])[:, :, keep], axis=2) for k in ["val_r2", "val_cc"]}

def _sp_sv(s, roi, good, z, folder=None):
    fn = (folder or SP_SESS) + f"regressPrfSplit_session_v{SP_VREG[roi]}_sub{s}{z}.pkl"
    if not os.path.exists(fn): return None
    with open(fn, "rb") as f: D = pickle.load(f)
    nsd = D["nsd"]; vi = np.concatenate([np.asarray(nsd["roiInd"][rn]).ravel() for rn in D["rois"]]); keep = good[vi]
    out = {}
    for key, name in [("r2oriSplit", "val_r2"), ("pearsonRori", "val_cc")]:
        M = np.concatenate([np.asarray(nsd[key][rn]) for rn in D["rois"]], axis=1)    # [data sess, vox, model sess]
        out[name] = np.nanmedian(M[:NSESS, keep, :NSESS], axis=1).T                  # -> [train, test]
    return out

SV_SOURCES = {   # all raw betas; SP controls only in Figure 5
    "SP": lambda s, r, g: _sp_sv(s, r, g, ""),
    "TS": _ts_sv,
    "SP 6 levels": lambda s, r, g: _sp_sv(s, r, g, "", SP_SESS_6LEV),
    "SP background": lambda s, r, g: _sp_sv(s, r, g, "", SP_SESS_BG),
}

def load_single_voxel(match):
    res = {}
    for s in SUBJECTS:
        goods = voxel_sets(s, match); print(f"single voxel, S{s}, matched={match}")
        for roi in ROIS:
            for m, fn in SV_SOURCES.items():
                d = fn(s, roi, goods[roi])
                if d is None: continue
                for k, v in d.items(): res.setdefault((m, roi, k), []).append(v)
    return {k: np.array(v) for k, v in res.items()}

# ---------- population / RDM drift (outputs of SP notebook 6 and TS save_perm_pop_TS) ----------
POP_MEASURES = {   # name: (file metric tag, matrix key, permutation key, similarity?)
    "Population response similarity": ("rdmCorrDist_cmpSpearman", "avgImgCorrMat", "imgCorrMatPerm", True),
    "RDM (1−r), Spearman":            ("rdmCorrDist_cmpSpearman", "betweenSessCorr", "betweenSessCorrPerm", True),
    "RDM (Euclidean), RMSE":          ("rdmEuclidean_cmpRMSE", "betweenSessCorr", "betweenSessCorrPerm", False),
    "RDM (1−r), RMSE":                ("rdmCorrDist_cmpRMSE", "betweenSessCorr", "betweenSessCorrPerm", False),
    "RDM (Euclidean), Spearman":      ("rdmEuclidean_cmpSpearman", "betweenSessCorr", "betweenSessCorrPerm", True),
}
def _pop_file(model, s, roi, tag, match):
    if model == "SP":
        return SP_SESS + ("matchVox/" if match else "") + f"permPop_N{NPERM}_v{SP_VREG[roi]}_sub{s}{SP_POP_ZSTR}_{tag}.npz", "Ori"
    return TS_POP + TS_EXCL + ("matchVox/" if match else "") + f"permPop_TS_{TS_TAG}_raw_N{NPERM}_S{s:02d}_{roi}_{tag}.npz", ""

def load_population(match, models=("SP", "TS")):
    """Per model x measure x ROI: per-subject matrices (no permutations needed: the group test re-permutes).
    The matched-voxel-count control is run for SP only: load_population(True, models=("SP",))."""
    res, missing = {}, []
    for model in models:
        for meas, (tag, key, pkey, is_sim) in POP_MEASURES.items():
            for roi in ROIS:
                Ms = []
                for s in SUBJECTS:
                    fn, suf = _pop_file(model, s, roi, tag, match)
                    if not os.path.exists(fn): missing.append(os.path.basename(fn)); continue
                    with np.load(fn, allow_pickle=True) as d: Ms.append(np.asarray(d[key + suf], float))
                if Ms: res[(model, meas, roi)] = np.array(Ms)
    if missing: print(f"population, matched={match}: {len(missing)} files missing, e.g. {missing[:3]}")
    return res


In [ ]:
# ==== Figure functions ====
from matplotlib.ticker import MaxNLocator

def _fig_legend(fig, models, colors, labels, y=1.0):
    handles = [plt.Line2D([], [], color=colors[m], lw=2.2) for m in models]
    fig.legend(handles, [labels[m] for m in models], loc="lower center", ncol=len(models),
               bbox_to_anchor=(0.5, y), fontsize=7.5, handlelength=1.8, columnspacing=1.6)

def _lag_axes(ax, title=None, ylabel=None):
    ax.set_xlim(0.5, NSESS - 0.5); ax.set_xticks([1, 10, 20, 29])
    ax.yaxis.set_major_locator(MaxNLocator(4))
    if title: ax.set_title(title)
    if ylabel: ax.set_ylabel(ylabel)

def _curve(ax, Gs, color, label):
    C = np.array([lag_curve(G) for G in Gs]); x = np.arange(1, NSESS)
    mu, se = np.nanmean(C, 0), sem(C)
    ax.fill_between(x, mu - se, mu + se, color=color, alpha=0.18, lw=0)
    ax.plot(x, mu, color=color, label=label)
    return mu, se

def _panel_label(ax, s):
    ax.text(-0.16, 1.06, s, transform=ax.transAxes, fontsize=10, fontweight="bold", va="bottom")

def _sig_text(ax, x, y, txt):
    ax.text(x, y, txt, ha="center", va="bottom", fontsize=7, color="#222222")

# ---------- Figure 1: whole-data validation R2, SP vs TS ----------
def figure1(W):
    fig, ax = plt.subplots(figsize=(W_FULL * 0.62, 2.6))
    series = [("SP (18), NSD pRFs", COL["SP"], LABEL["SP"]), ("TS k1 (58), NSD pRFs", COL["TS"], LABEL["TS"])]
    xs = np.arange(4)
    pos = bars_with_subjects(ax, xs, [[W[m][:, ri] for ri in range(4)] for m, _, _ in series],
                             [c for _, c, _ in series], [l for _, _, l in series])
    her = W.get("Henderson et al. (2023), TS")
    if her is not None and np.isfinite(her).any():                          # published reference, same voxels
        for ri in range(4):
            ax.plot([xs[ri] - 0.38, xs[ri] + 0.38], [np.nanmean(her[:, ri])] * 2, color="#222222", lw=1.0, ls=(0, (3, 2)), zorder=5)
        ax.plot([], [], color="#222222", lw=1.0, ls=(0, (3, 2)), label="Henderson et al. (2023), TS")
    rows, ps = [], []
    for ri, roi in enumerate(ROIS):
        a, b = W["TS k1 (58), NSD pRFs"][:, ri], W["SP (18), NSD pRFs"][:, ri]; ok = np.isfinite(a) & np.isfinite(b)
        from scipy.stats import wilcoxon
        p = wilcoxon(a[ok] - b[ok]).pvalue if ok.sum() > 5 else np.nan; ps.append(p)
        for m in W: rows.append(dict(model=m, roi=roi, mean=np.nanmean(W[m][:, ri]), sem=sem(W[m][:, ri]),
                                     **{f"S{s}": W[m][si, ri] for si, s in enumerate(SUBJECTS)}))
    q = fdr_bh(ps)
    top = np.nanmax([W[m] for m in W if not m.startswith("TS, H")]) * 1.06
    for ri in range(4): _sig_text(ax, xs[ri], top, stars(q[ri]) if np.isfinite(q[ri]) else "")
    ax.set_ylabel("validation R² (whole-data fit)"); ax.set_ylim(0, top * 1.12)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, fontsize=7)
    meta = dict(figure="Whole-data validation R2, SP vs TS",
                voxels="Henderson Fig. 3B rule: ROI voxels with noise ceiling > 0.01, category-ROI overlap removed; OLS voxels with R2 < -1 excluded",
                fit="all sessions, averaged repetitions, validation on shared-1000 images, OLS, NSD pRFs (TS: aligned to grid 5)",
                test="paired Wilcoxon TS vs SP per ROI across subjects, FDR-BH over 4 ROIs", q_TS_vs_SP=dict(zip(ROIS, q)))
    save_fig(fig, "Fig1_wholedata_SP_vs_TS", meta, rows); plt.show()
    print("Caption draft: Validation R² of whole-data fits for the SP (blue) and TS k1 (orange) models with NSD pRFs. "
          "Bars, mean across subjects; error bars, SEM; dots, subjects; dashed lines, Henderson et al. (2023) on the same voxels. "
          "Stars, paired Wilcoxon TS vs SP, FDR-corrected across ROIs.")

def figure1_validation(W):
    """Supplementary: implementation check against Henderson et al. (2023)."""
    fig, ax = plt.subplots(figsize=(W_FULL * 0.75, 2.6))
    series = [("Henderson et al. (2023), TS", COL["Henderson"], "Henderson et al. (2023), her pRFs"),
              ("TS, Henderson pRFs (validation)", COL["TS_herpRF"], "This study, TS, Henderson pRFs"),
              ("TS k1 (58), NSD pRFs", COL["TS"], "This study, TS k1, NSD pRFs"),
              ("SP (18), NSD pRFs", COL["SP"], "This study, SP, NSD pRFs")]
    bars_with_subjects(ax, np.arange(4), [[W[m][:, ri] for ri in range(4)] for m, _, _ in series],
                       [c for _, c, _ in series], [l for _, _, l in series], width=0.2)
    ax.set_ylabel("validation R² (whole-data fit)"); ax.legend(fontsize=6.8, loc="upper right")
    rows = [dict(model=m, roi=roi, mean=np.nanmean(W[m][:, ri]), sem=sem(W[m][:, ri])) for m in W for ri, roi in enumerate(ROIS)]
    save_fig(fig, "FigS1_validation_Henderson", dict(figure="Implementation check against Henderson et al. (2023)"), rows); plt.show()

# ---------- Figure 2: single-voxel drift, SP vs TS ----------
def figure2(SV, SVm=None, models=("SP", "TS"), name="Fig2_single_voxel_SP_vs_TS", colors=None, labels=None):
    colors = colors or {m: COL[m] for m in models}; labels = labels or {m: LABEL.get(m, m) for m in models}
    fig = plt.figure(figsize=(W_FULL, 5.6))
    gs = fig.add_gridspec(3, 4, height_ratios=[1, 1, 1.05], hspace=0.62, wspace=0.55)
    rows, stats = [], {}
    for ri, roi in enumerate(ROIS):
        for row, (k, ylab) in enumerate([("val_r2", "cross-session cvR²"), ("val_cc", "cross-session r")]):
            ax = fig.add_subplot(gs[row, ri])
            for m in models:
                if (m, roi, k) not in SV: continue
                mu, se = _curve(ax, SV[(m, roi, k)], colors[m], labels[m])
                for d in range(NSESS - 1): rows.append(dict(panel=k, model=m, roi=roi, lag=d + 1, mean=mu[d], sem=se[d]))
            _lag_axes(ax, roi if row == 0 else None, ylab if ri == 0 else None)
            if row == 1: ax.set_xlabel("session lag")
            if ri == 0 and row == 0: _panel_label(ax, "A")
            if ri == 0 and row == 1: _panel_label(ax, "B")
    # drift strength per ROI (cvR2), all voxels and matched voxel count
    for j, (D, title) in enumerate([(SV, "all voxels"), (SVm, "matched voxel count")]):
        if D is None: continue
        ax = fig.add_subplot(gs[2, :] if SVm is None else gs[2, 2 * j:2 * j + 2])
        if SVm is None: title = ""
        vals, qs = [], {}
        for m in models:
            res = [group_test(D[(m, roi, "val_r2")], -1) if (m, roi, "val_r2") in D else None for roi in ROIS]
            q = fdr_bh([r["p"] if r else 1 for r in res]); qs[m] = q
            vals.append([r["subj"] if r else None for r in res])
            for roi, r, qq in zip(ROIS, res, q):
                if r: stats[(title, m, roi)] = dict(mean=r["mean"], p=r["p"], q=qq); rows.append(dict(panel=f"drift {title}", model=m, roi=roi, lag="", mean=r["mean"], sem=sem(r["subj"])))
        pos = bars_with_subjects(ax, np.arange(4), vals, [colors[m] for m in models], [labels[m] for m in models])
        ymax = np.nanmax([np.nanmax(v) for vv in vals for v in vv if v is not None])
        for p_, m in zip(pos, models):
            for x, qq in zip(p_, qs[m]): _sig_text(ax, x, ymax * 1.04, stars(qq))
        ax.set_ylim(min(0, ax.get_ylim()[0]), ymax * 1.2); ax.set_title(title)
        ax.set_ylabel("drift strength\n(−r, cvR² vs lag)" if j == 0 else "")
        if j == 0: _panel_label(ax, "C")
    _fig_legend(fig, models, colors, labels, y=0.93)
    meta = dict(figure="Single-voxel cross-session drift", betas="raw (no within-session normalization)",
                models={m: labels[m] for m in models}, voxels="ROI voxels with NSD pRF R2 > 0; matched: each ROI subsampled to the subject's smallest ROI (seed [0, subject, ROI])",
                aggregation="median across voxels per session pair (30 x 30 per subject); curves mean ± SEM across subjects",
                drift="-Pearson r(cvR2, lag) over off-diagonal entries, per subject, mean across subjects",
                test="1000 session-order permutations (same for all subjects), one-sided; FDR-BH over 4 ROIs per model",
                stats={f"{a}|{b}|{c}": v for (a, b, c), v in stats.items()})
    save_fig(fig, name, meta, rows); plt.show()
    print("Caption draft: (A) Cross-session cvR² and (B) Pearson r as a function of session lag (median across voxels, mean ± SEM across subjects). "
          "(C) Drift strength (negative correlation of cvR² with lag) per ROI; bars, mean; dots, subjects; stars, session-order permutation test, FDR-corrected across ROIs.")
    return stats

# ---------- Figures 3 and 4: population and RDM drift, SP vs TS ----------
def figure_pop(P, Pm, measures, name, ylab_curve, caption):
    n = len(measures)
    fig = plt.figure(figsize=(W_FULL, 2.3 * n + 0.6))
    gs = fig.add_gridspec(n, 6, hspace=0.95, wspace=0.75)
    rows, stats = [], {}
    for i, meas in enumerate(measures):
        is_sim = POP_MEASURES[meas][3]
        D = Pm if Pm else P                    # lag curves and bars: matched voxel count if available
        sub = gs[i, 0:4].subgridspec(1, 4, wspace=0.62)
        for ri, roi in enumerate(ROIS):
            ax = fig.add_subplot(sub[0, ri])
            for m in ["SP", "TS"]:
                if (m, meas, roi) in D:
                    mu, se = _curve(ax, D[(m, meas, roi)], COL[m], LABEL[m])
                    for d in range(NSESS - 1): rows.append(dict(measure=meas, panel="curve", model=m, roi=roi, lag=d + 1, mean=mu[d], sem=se[d]))
            _lag_axes(ax, roi, ylab_curve[is_sim] if ri == 0 else None)
            ax.set_xlabel("session lag")
            if ri == 0:
                _panel_label(ax, "ABCDEFGH"[2 * i])
                if n > 1: ax.text(-0.16, 1.32, meas, transform=ax.transAxes, fontsize=9, fontweight="bold", va="bottom")
        ax = fig.add_subplot(gs[i, 4:6])
        vals, qs = [], {}
        for m in ["SP", "TS"]:
            res = [group_test(D[(m, meas, roi)], -1 if is_sim else 1) if (m, meas, roi) in D else None for roi in ROIS]
            q = fdr_bh([r["p"] if r else 1 for r in res]); qs[m] = q
            vals.append([r["subj"] if r else None for r in res])
            for roi, r, qq in zip(ROIS, res, q):
                if r:
                    stats[(meas, m, roi)] = dict(mean=r["mean"], p=r["p"], q=qq)
                    rows.append(dict(measure=meas, panel="drift", model=m, roi=roi, lag="", mean=r["mean"], sem=sem(r["subj"])))
        pos = bars_with_subjects(ax, np.arange(4), vals, [COL["SP"], COL["TS"]], [LABEL["SP"], LABEL["TS"]])
        finite = [np.nanmax(v) for vv in vals for v in vv if v is not None]
        ymax = max(finite) if finite else 1
        for p_, m in zip(pos, ["SP", "TS"]):
            for x, qq in zip(p_, qs[m]): _sig_text(ax, x, ymax * 1.04, stars(qq) if (m, meas, ROIS[0]) in D else "")
        ax.set_ylim(min(0, ax.get_ylim()[0]), ymax * 1.22); ax.set_ylabel("drift strength")
        _panel_label(ax, "ABCDEFGH"[2 * i + 1])
    _fig_legend(fig, ["SP", "TS"], COL, LABEL, y=1.02 if n == 1 else 1.0)
    meta = dict(figure=name, measures=measures, voxels="matched voxel count" if Pm else "all voxels",
                source="SP notebook 6 (raw, full SP model) and TS save_perm_pop_TS (k1, raw); 100 simulated images per session",
                drift="lag correlation, signed so that positive = more drift (-r for similarity, +r for RMSE), per subject; mean across subjects",
                test="1000 session-order permutations (same for all subjects), one-sided; FDR-BH over 4 ROIs per model and measure",
                stats={f"{a}|{b}|{c}": v for (a, b, c), v in stats.items()})
    save_fig(fig, name, meta, rows); plt.show(); print("Caption draft:", caption)
    return stats

def figure3(P, Pm, suffix=""):
    return figure_pop(P, Pm, ["Population response similarity"], "Fig3_population_SP_vs_TS" + suffix,
                      {True: "similarity (r)", False: ""},
                      "(A) Similarity between simulated population responses to the same image in two sessions as a function of session lag "
                      "(mean ± SEM across subjects). (B) Drift strength per ROI; bars, mean; dots, subjects; stars, permutation test, FDR-corrected.")

def figure4(P, Pm, suffix=""):
    return figure_pop(P, Pm, ["RDM (1−r), Spearman", "RDM (Euclidean), RMSE"], "Fig4_RDM_SP_vs_TS" + suffix,
                      {True: "Spearman ρ", False: "RMSE"},
                      "(A, C) Similarity (Spearman) and distance (RMSE) between session RDMs as a function of session lag; (B, D) drift strength per ROI. "
                      "Rank-based comparison tests relational geometry; RMSE is also sensitive to the overall scale of the distances.")

# ---------- Figure 5: SP controls ----------
def figure5(SV, SVm=None):
    """SP controls: matched voxel count, 4 x 6 SF levels, gray background (all single voxel)."""
    SV = dict(SV)
    if SVm:
        for roi in ROIS:
            for k in ("val_r2", "val_cc"):
                if ("SP", roi, k) in SVm: SV[("SP matched voxels", roi, k)] = SVm[("SP", roi, k)]
    ctrl = ["SP", "SP matched voxels", "SP 6 levels", "SP background"]
    present = [m for m in ctrl if any((m, r, "val_r2") in SV for r in ROIS)]
    n_subj = {m: len(SV[(m, "V1", "val_r2")]) for m in present if (m, "V1", "val_r2") in SV}
    print("subjects available per SP version:", n_subj)
    labels = {"SP": "SP 4 × 4 (18)", "SP matched voxels": "SP 4 × 4, matched voxel count", "SP 6 levels": "SP 4 × 6 (26)", "SP background": "SP 4 × 4, gray background"}
    labels = {m: labels[m] + (f" (n={n_subj[m]})" if n_subj.get(m, 8) < 8 else "") for m in present}
    return figure2(SV, None, models=tuple(present), name="Fig5_SP_controls",
                   colors={m: COL[m] for m in present}, labels=labels)


## Figure 1 — whole-data validation R², SP vs TS (+ supplementary implementation check)

In [ ]:
W = cached("wholedata", load_wholedata)
figure1(W)
figure1_validation(W)

## Figure 2 — single-voxel drift, SP vs TS

In [ ]:
SV  = cached("single_voxel_all", lambda: load_single_voxel(False))
SVm = cached("single_voxel_matched", lambda: load_single_voxel(True))
stats_sv = figure2(SV)            # SP vs TS, all voxels

## Figures 3 and 4 — population and RDM drift, SP vs TS
Main figures use all voxels for both models. The matched-voxel-count control is run for SP only (supplementary `_SP_matchedvoxels`).

In [ ]:
P = load_population(False)                     # SP and TS, all voxels (main figures)
Pm_SP = load_population(True, models=("SP",)) # matched voxel count: SP only
stats_pop = figure3(P, None)
stats_rdm = figure4(P, None)
# supplementary: SP with matched voxel count (TS not run matched)
figure3(Pm_SP, None, suffix="_SP_matchedvoxels")
figure4(Pm_SP, None, suffix="_SP_matchedvoxels")

## Figure 5 — SP controls (matched voxel count; 4 × 6 SF levels; images on a gray background)
The background version is shown for the subjects whose session-wise regression exists (n is given in the legend).

In [ ]:
stats_ctrl = figure5(SV, SVm)